In [5]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Load datasets
X = pd.read_excel('../minmax.xlsx')
y = pd.read_csv('../idC_with_header.csv')['Label']

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define fitness function
def fitness_function(params, model_type='svm'):
    if model_type == 'svm':
        C, gamma = params
        model = SVC(C=C, gamma=gamma)
    elif model_type == 'rf':
        n_estimators, max_depth = params
        model = RandomForestClassifier(n_estimators=int(n_estimators), max_depth=int(max_depth))
    else:
        raise ValueError("Unsupported model type.")
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    return accuracy_score(y_test, predictions)

# GSA parameters
num_agents = 10
max_iter = 20
G0 = 100
alpha = 20

# Hyperparameter bounds
svm_bounds = {'C': (0.1, 100), 'gamma': (0.0001, 1)}
rf_bounds = {'n_estimators': (10, 200), 'max_depth': (1, 50)}

# Initialize agents
def initialize_agents(bounds):
    agents = []
    for _ in range(num_agents):
        agent = []
        for key in bounds:
            low, high = bounds[key]
            agent.append(np.random.uniform(low, high))
        agents.append(agent)
    return np.array(agents)

# GSA optimization
def gsa_optimization(bounds, model_type='svm'):
    agents = initialize_agents(bounds)
    velocities = np.zeros_like(agents)
    best_agent = None
    best_fitness = -np.inf

    for t in range(max_iter):
        fitness = np.array([fitness_function(agent, model_type) for agent in agents])
        best_idx = np.argmax(fitness)
        if fitness[best_idx] > best_fitness:
            best_fitness = fitness[best_idx]
            best_agent = agents[best_idx].copy()

        worst = np.min(fitness)
        best = np.max(fitness)
        mass = (fitness - worst) / (best - worst + 1e-10)
        mass = mass / mass.sum()

        G = G0 * np.exp(-alpha * t / max_iter)

        for i in range(num_agents):
            force = np.zeros(agents.shape[1])
            for j in range(num_agents):
                if i != j:
                    distance = np.linalg.norm(agents[j] - agents[i]) + 1e-10
                    force += np.random.rand() * G * (mass[i] * mass[j]) * (agents[j] - agents[i]) / distance
            acceleration = force / (mass[i] + 1e-10)
            velocities[i] = np.random.rand() * velocities[i] + acceleration
            agents[i] += velocities[i]

            # Ensure agents stay within bounds
            for k, key in enumerate(bounds):
                low, high = bounds[key]
                agents[i][k] = np.clip(agents[i][k], low, high)

    return best_agent

# Optimize SVM
best_svm_params = gsa_optimization(svm_bounds, model_type='svm')
best_C, best_gamma = best_svm_params
svm_model = SVC(C=best_C, gamma=best_gamma)
svm_model.fit(X_train, y_train)
svm_predictions = svm_model.predict(X_test)

# Evaluate SVM
print("SVM Performance:")
print("Accuracy:", accuracy_score(y_test, svm_predictions))
print("Precision:", precision_score(y_test, svm_predictions, average='weighted'))
print("Recall:", recall_score(y_test, svm_predictions, average='weighted'))
print("F1 Score:", f1_score(y_test, svm_predictions, average='weighted'))
print("Best Hyperparameters:", {'C': best_C, 'gamma': best_gamma})

# Optimize RF
best_rf_params = gsa_optimization(rf_bounds, model_type='rf')
best_n_estimators, best_max_depth = best_rf_params
rf_model = RandomForestClassifier(n_estimators=int(best_n_estimators), max_depth=int(best_max_depth))
rf_model.fit(X_train, y_train)
rf_predictions = rf_model.predict(X_test)

# Evaluate RF
print("\nRandom Forest Performance:")
print("Accuracy:", accuracy_score(y_test, rf_predictions))
print("Precision:", precision_score(y_test, rf_predictions, average='weighted'))
print("Recall:", recall_score(y_test, rf_predictions, average='weighted'))
print("F1 Score:", f1_score(y_test, rf_predictions, average='weighted'))
print("Best Hyperparameters:", {'n_estimators': int(best_n_estimators), 'max_depth': int(best_max_depth)})


SVM Performance:
Accuracy: 0.9101123595505618
Precision: 0.9386837881219904
Recall: 0.9101123595505618
F1 Score: 0.9113891726251276
Best Hyperparameters: {'C': 45.23055926177988, 'gamma': 0.02942193636614268}

Random Forest Performance:
Accuracy: 0.9438202247191011
Precision: 0.9454064771976206
Recall: 0.9438202247191011
F1 Score: 0.9442257328715046
Best Hyperparameters: {'n_estimators': 79, 'max_depth': 18}


GSA-Based Feature Selection for Naive Bayes Classification

In [6]:
# Import necessary libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.naive_bayes import GaussianNB

# Load the dataset
X = pd.read_excel('../minmax.xlsx')
y = pd.read_csv('../idC_with_header.csv')['Label']

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the fitness function for GSA
def fitness_function(position):
    # Select features based on the binary position vector
    selected_features = [i for i in range(len(position)) if position[i] > 0.5]
    if len(selected_features) == 0:  # Avoid empty feature subsets
        return 0
    
    # Train a GaussianNB model
    model = GaussianNB()
    model.fit(X_train.iloc[:, selected_features], y_train)
    
    # Evaluate the model on the test set
    y_pred = model.predict(X_test.iloc[:, selected_features])
    accuracy = accuracy_score(y_test, y_pred)
    return accuracy

# Initialize GSA parameters
num_agents = 20
num_features = X.shape[1]  # Number of features in the dataset
max_iterations = 25
G = 100  # Gravitational constant
epsilon = 1e-5  # Small value to avoid division by zero

# Initialize agents' positions randomly (binary positions for feature selection)
agents = np.random.randint(0, 2, size=(num_agents, num_features))
velocities = np.zeros_like(agents)

# GSA algorithm
for iteration in range(max_iterations):
    fitness = np.array([fitness_function(agent) for agent in agents])
    
    # Calculate masses based on fitness
    masses = (fitness - fitness.min()) / (fitness.max() - fitness.min() + epsilon)
    masses /= masses.sum()
    
    # Update positions and velocities
    for i in range(num_agents):
        force = np.zeros(num_features)
        for j in range(num_agents):
            if i != j:
                distance = np.linalg.norm(agents[i] - agents[j]) + epsilon
                force += G * masses[j] * (agents[j] - agents[i]) / distance
        acceleration = force / (masses[i] + epsilon)
        velocities[i] = np.random.rand() * velocities[i] + acceleration
        agents[i] = agents[i] + velocities[i]
        agents[i] = np.clip(agents[i], 0, 1)
        agents[i] = np.round(agents[i])  # Convert to binary

# Evaluate the best solution
best_agent = agents[np.argmax(fitness)]
selected_features = [i for i in range(len(best_agent)) if best_agent[i] > 0.5]

if len(selected_features) == 0:
    print("No features selected by GSA. Using all features as fallback.")
    selected_features = list(range(X.shape[1]))  # fallback: use all features

# Train the final model with GaussianNB using the selected features
final_model = GaussianNB()
final_model.fit(X_train.iloc[:, selected_features], y_train)
y_pred = final_model.predict(X_test.iloc[:, selected_features])

# Print performance metrics
print(f"Selected Features: {selected_features}")
print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Selected Features: [2, 4, 5, 6, 9, 10, 14, 15, 18, 21, 23, 24, 25, 26, 27, 28, 30, 33, 35, 36, 37, 39, 40, 42, 44, 45, 47, 48, 51, 52, 57, 58, 59, 60, 62, 63, 68, 69, 75, 76, 77, 79, 80, 85, 88, 89, 91, 92, 93, 97, 98, 99, 101, 103, 104, 105, 106, 107, 108, 109, 110, 112, 114, 117, 119, 121, 122, 123, 125, 128, 129, 130, 132, 133, 135, 136, 137, 138, 141, 143, 145, 146, 147, 149, 152, 154, 155, 156, 160, 161, 163, 167, 170, 171, 173, 174, 175, 176, 178, 180, 188, 192, 193, 194, 195, 197, 198, 200, 204, 207, 208, 211, 213, 214, 215, 217, 222, 223, 226, 228, 232, 234, 237, 238, 239, 240, 242, 243, 244, 245, 246, 260, 261, 264, 266, 267, 270, 272, 275, 277, 278, 279, 280, 282, 283, 285, 286, 287, 288, 291, 292, 293, 294, 295, 296, 298, 299, 301, 302, 303, 304, 305, 306, 308, 312, 313, 315, 317, 318, 319, 320, 321, 324, 326, 328, 331, 332, 333, 337, 339, 341, 342, 345, 347, 348, 349, 350, 355, 358, 359, 360, 361, 362, 363, 365, 366, 367, 368, 370, 371, 374, 379, 380, 381, 387, 388, 390, 39